# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EarlPly/Start-W1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

**Lane carried forward from ML-02:** Structured Content Archetype Clustering. This is a framing exercise on the anonymized starter snapshot, not a trained model or a claim about Google's ranking system. The intended user is an editor or SEO analyst deciding which pages to inspect and what kind of review to do next.


## 1. My lane as an ML task (type)

**Task type: unsupervised clustering.** Group content pages by *observed* search and engagement patterns, then describe each group in plain language. One row represents one pseudonymized content page in a trailing 90-day snapshot. The output would be a cluster assignment and an evidence summary, not an automatic instruction to rewrite or remove a page.

The decision is **which pages merit review, and which review playbook fits**. An editor could inspect high-impression, low-CTR pages for title/snippet issues; inspect aging, declining pages for possible refreshes; and monitor stable performers. A low-reach page is a **research candidate**, never an automatic prune or redirect. That decision needs query context and human review. The starter data contains 30,000 distinct pages across 32 pseudonymized clients, enough to explore whether there are recurring patterns.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

relative_path = Path('data/raw/content_refresh_anonymized.csv')
if not relative_path.exists():
    relative_path = Path('../../data/raw/content_refresh_anonymized.csv')
source = (str(relative_path) if relative_path.exists() else
          'https://raw.githubusercontent.com/EarlPly/Start-W1/main/data/raw/content_refresh_anonymized.csv')
pages = pd.read_csv(source)
assert pages['content_id'].is_unique, 'Expected one row per content page'
print(f'Observed pages: {len(pages):,}; pseudonymized clients: {pages.client_id.nunique():,}')
print('Content types:', pages.content_type.value_counts().to_dict())


Observed pages: 30,000; pseudonymized clients: 32
Content types: {'keyword article': 27207, 'feedly article': 2096, 'comparison article': 697}


## 2. Target or proxy

There is **no supervised target column** for clustering. A future `archetype_cluster` column would hold a *model-assigned segment*, not an observed truth label. The input proxies are measured page signals: 90-day impressions (log-scaled), CTR, average search position where available, engagement rate, content age, time since last update, and a 30-day impression trend. These describe current patterns, not the benefit of an edit. I would scale numeric fields, handle missing values explicitly, inspect correlated measures, and compare profiles within or across content types before choosing a clustering method.

The starter snapshot cannot reveal whether a refresh *caused* a recovery. For a later supervised question, the target would have to be a separately observed future outcome from a later time window in the warehouse data. I will not relabel `trend_direction` or `trend_pct` as a future target, because both are calculated from this snapshot's recent impressions.


In [2]:
proxy_columns = ['impressions_90d', 'ctr', 'avg_position', 'engagement_rate',
                 'content_age_days', 'days_since_last_update', 'trend_pct']
proxy_audit = pd.DataFrame({
    'missing_rows': pages[proxy_columns].isna().sum(),
    'observed_median': pages[proxy_columns].median(numeric_only=True),
})
print('Candidate proxy audit (rates are percentage points, not fractions):')
display(proxy_audit)
print(f"Position value 0 means unavailable for {(pages.avg_position == 0).sum():,} pages; it is not rank zero.")


Candidate proxy audit (rates are percentage points, not fractions):
Position value 0 means unavailable for 1,205 pages; it is not rank zero.


,missing_rows,observed_median
impressions_90d,0,731.00
ctr,0,0.07
avg_position,0,10.80
engagement_rate,0,0.00
content_age_days,0,236.00
days_since_last_update,0,20.00
trend_pct,3388,-33.50


## 3. Success metric

**Primary framing metric for the later cluster experiment: adjusted Rand index (ARI) for stability.** Fit the same clustering workflow on two independently drawn 80% page samples, then compare assignments for pages in their overlap. ARI ignores arbitrary cluster-number names; 1 means identical partitions and about 0 means chance-level agreement. I will provisionally call **ARI ≥ 0.70** useful stability, provided no cluster is merely a missing-data or content-type bucket. This threshold is a *predeclared evaluation criterion*, not a result measured in this notebook.

Stability alone is insufficient for the editorial decision. Before recommending an action, an editor must be able to describe each cluster with measured signals and agree that its suggested review is sensible on a held-out sample. A wrong refresh wastes editorial time; a wrong prune/redirect can lose useful traffic. Thus every cluster output remains decision support, with manual review required. I will compare it with a transparent fixed-rule grouping in later work.


In [3]:
eligible = pages[(pages['impressions_90d'] >= 100) & (pages['avg_position'] > 0)]
print(f'Pages with at least 100 impressions and a measured position: {len(eligible):,} / {len(pages):,}')
print(f'Pages held aside for low-volume or unavailable-position review: {len(pages) - len(eligible):,}')
print('These counts define a possible evaluation population; no ARI has been computed yet.')


Pages with at least 100 impressions and a measured position: 22,006 / 30,000
Pages held aside for low-volume or unavailable-position review: 7,994
These counts define a possible evaluation population; no ARI has been computed yet.


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content page in one trailing 90-day snapshot.** The ID is only a grouping key, never a model feature. The dataframe below shows the actual lane slice. It preserves low-volume pages for descriptive review; the `cluster_eligible` flag marks the cleaner subset used for a first position-aware clustering experiment. A page with fewer than 100 impressions is not automatically bad content, and `avg_position == 0` means no position data.

The proposed `archetype_cluster` column is sketched as missing because no clustering model has been fit. Filling it with a hand-written rule would turn a model output into an invented label.


In [4]:
lane_columns = ['content_id', 'content_type', 'main_intent', 'impressions_90d',
                'clicks_90d', 'ctr', 'avg_position', 'engagement_rate',
                'content_age_days', 'days_since_last_update', 'trend_pct']
lane_df = pages[lane_columns].copy()
lane_df['cluster_eligible'] = ((lane_df.impressions_90d >= 100) &
                               (lane_df.avg_position > 0))
lane_df['archetype_cluster'] = pd.Series(pd.NA, index=lane_df.index, dtype='string')
assert len(lane_df) == lane_df.content_id.nunique()
print(f'Lane dataframe: {lane_df.shape[0]:,} rows × {lane_df.shape[1]} columns')
display(lane_df.head(8))


Lane dataframe: 30,000 rows × 13 columns


,content_id,content_type,main_intent,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,content_age_days,days_since_last_update,trend_pct,cluster_eligible,archetype_cluster
0,content_304f48230142,keyword article,transactional,3803,29,0.76,10.6,5.88,187,20,-41.4,True,<NA>
1,content_a1fb4e703a9e,keyword article,informational,15320,7,0.05,20.3,0.00,445,25,-57.7,True,<NA>
2,content_9aa793d4d895,keyword article,informational,12581,11,0.09,36.5,0.00,141,20,-60.9,True,<NA>
3,content_331d6c4de07b,keyword article,commercial,11751,58,0.49,6.2,1.28,463,22,-13.8,True,<NA>
4,content_d99b7a2d90ca,keyword article,informational,19140,24,0.13,44.0,0.00,263,14,-34.7,True,<NA>
5,content_d4084a4bc775,keyword article,transactional,3970,1,0.03,8.5,0.00,147,20,-38.9,True,<NA>
6,content_9a34b442b552,keyword article,informational,20,0,0.00,7.0,0.00,90,20,-92.3,False,<NA>
7,content_a63219c6e95a,keyword article,commercial,1724,1,0.06,21.2,3.57,445,22,0.6,True,<NA>


## 5. Why ML beats a fixed rule here

A fixed rule such as “CTR below 0.2% means rewrite” is a useful baseline, but it conflates pages with different search positions, volumes, ages, and trends. In the actual data, even pages meeting that same rule span varied positions and trends (checked below). Clustering may surface *combinations* that help route pages to a more fitting review, rather than treating every weak CTR as the same problem.

ML has to earn its place: if the clusters are unstable, mirror missingness or content type, or do not change an editor's review decision beyond a simple rule, I would use the rule or a dashboard instead. The output can support title/snippet review, refresh investigation, consolidation research, or monitoring; it cannot prove what caused performance or justify automatic deletion.


In [5]:
rule_slice = pages[(pages.impressions_90d >= 100) & (pages.ctr < 0.2) &
                   (pages.avg_position > 0)]
print(f'Pages matching the simple low-CTR rule: {len(rule_slice):,}')
print('Within that same rule, average-position quartiles:')
print(rule_slice.avg_position.quantile([0.25, 0.50, 0.75]).round(1).to_string())
print('Within that same rule, trend mix:')
print(rule_slice.trend_direction.value_counts().to_string())


Pages matching the simple low-CTR rule: 12,847
Within that same rule, average-position quartiles:
0.25     8.5
0.50    16.1
0.75    29.2
Within that same rule, trend mix:
trend_direction
down      8146
stable    2605
up        1938
new        131
flat        27


## Self-check

- [x] Each section contains written framing and code-backed checks from the real starter data.
- [x] Code cells run top to bottom without errors; outputs are saved below them.
- [x] No client names, URLs, raw queries, or private data appear in the outputs.
- [x] Claims are limited to observed, measured, directional, decision-support findings.
- [x] The target sketch is explicitly unsupervised; no current-window label is passed off as a future outcome.
- [x] Notebook is ready to commit under `work/notebooks/` and submit as the repository URL.
